In [5]:
import json
import csv
import os

# File paths (set for the Kaggle environment)
INPUT_JSON = '/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json'
OUTPUT_DIR = '/kaggle/working/'

# Target categories we want to extract
TARGET_CATEGORIES = {'cs.DB', 'cs.AI'}

def process_arxiv_data():
    # Temporary data structures to avoid duplicate authors
    seen_authors = set()
    
    # Open CSV files for writing
    with open(os.path.join(OUTPUT_DIR, 'papers.csv'), 'w', encoding='utf-8', newline='') as f_papers, \
         open(os.path.join(OUTPUT_DIR, 'authors.csv'), 'w', encoding='utf-8', newline='') as f_authors, \
         open(os.path.join(OUTPUT_DIR, 'wrote.csv'), 'w', encoding='utf-8', newline='') as f_wrote, \
         open(os.path.join(OUTPUT_DIR, 'paper_categories.csv'), 'w', encoding='utf-8', newline='') as f_cat:
        
        # Initialize CSV writers
        writer_papers = csv.writer(f_papers)
        writer_authors = csv.writer(f_authors)
        writer_wrote = csv.writer(f_wrote)
        writer_cat = csv.writer(f_cat)
        
        # Write headers
        writer_papers.writerow(['paper_id', 'title', 'year'])
        writer_authors.writerow(['author_id', 'name'])
        writer_wrote.writerow(['author_id', 'paper_id'])
        writer_cat.writerow(['paper_id', 'category'])
        
        # Counter to print progress
        papers_kept = 0
        
        print("Starting file processing (this operation will take a few minutes but use very little RAM)...\n\n")
        
        # Read line by line to avoid saturating RAM
        with open(INPUT_JSON, 'r', encoding='utf-8') as f_in:
            for line in f_in:
                paper = json.loads(line)
                
                # Categories in the JSON are a space-separated string, e.g.: "cs.AI cs.LG stat.ML"
                paper_categories = set(paper.get('categories', '').split())
                
                # Check if the paper belongs to at least one of the target categories
                if not TARGET_CATEGORIES.isdisjoint(paper_categories):
                    papers_kept += 1
                    
                    paper_id = paper['id']
                    
                    # Clean the title (remove newlines to avoid issues in the CSVs)
                    title = paper['title'].replace('\n', ' ').replace('"', "'").strip()
                    
                    # Extract the year (the first two characters of the ID, or use update_date)
                    year = paper.get('update_date', '2000-01-01').split('-')[0]
                    
                    # 1. Write the paper
                    writer_papers.writerow([paper_id, title, year])
                    
                    # 2. Write the categories (Paper -> Category relation)
                    for cat in paper_categories:
                        if cat in TARGET_CATEGORIES: # Filtriamo solo cs.DB e cs.AI per pulizia
                            writer_cat.writerow([paper_id, cat])
                    
                    # 3. Handle authors (the JSON has 'authors_parsed' which is a list of lists)
                    # E.g.: [["Last Name", "First Name", "Suffix"], ...]
                    for author_parts in paper.get('authors_parsed', []):
                        if len(author_parts) >= 2:
                            last_name = author_parts[0]
                            first_name = author_parts[1]
                            # Create an empirical unique ID (clean_first_last_name)
                            full_name = f"{first_name} {last_name}".strip()
                            author_id = full_name.lower().replace(' ', '_').replace('.', '')
                            
                            # If it's a new author, add it to authors.csv
                            if author_id not in seen_authors:
                                writer_authors.writerow([author_id, full_name])
                                seen_authors.add(author_id)
                            
                            # Write the relation (Edge/Bridge Table WROTE between Author and Paper)
                            writer_wrote.writerow([author_id, paper_id])

        print(f"Processing complete! Found and saved {papers_kept} papers for cs.DB and cs.AI.")
        print(f"Total unique authors extracted: {len(seen_authors)}")

# Execute the function
process_arxiv_data()


Starting file processing (this operation will take a few minutes but use very little RAM)...


Processing complete! Found and saved 198719 papers for cs.DB and cs.AI.
Total unique authors extracted: 329615
